# Task 4: Advanced Analytics (Basic)

> **ApexPlanet Data Analytics Internship**  
> *30-Day Program | Day 21-26*

## What You'll Learn
- Day 21-22: Statistical Analysis (Descriptive stats, Hypothesis testing, Confidence intervals)
- Day 23-24: Time Series Analysis & Customer Segmentation (K-Means Clustering)
- Day 25-26: Basic Predictive Modeling (Linear Regression, Logistic Regression)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ All libraries imported!")

In [ ]:
# Load cleaned data
df = pd.read_csv('../data/superstore_cleaned.csv')
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

## Day 21-22: Descriptive Statistics

Calculate mean, median, mode, std dev, skewness, kurtosis

In [ ]:
numeric_cols = ['Sales', 'Quantity', 'Discount', 'Profit']

print("DESCRIPTIVE STATISTICS:")
for col in numeric_cols:
    print(f"\n{col}:")
    print(f"  Mean: {df[col].mean():.2f}")
    print(f"  Median: {df[col].median():.2f}")
    print(f"  Std Dev: {df[col].std():.2f}")
    print(f"  Skewness: {df[col].skew():.2f}")
    print(f"  Kurtosis: {df[col].kurtosis():.2f}")

## Hypothesis Testing

### T-Test: Compare Sales between Technology and Furniture

In [ ]:
tech_sales = df[df['Category'] == 'Technology']['Sales']
furniture_sales = df[df['Category'] == 'Furniture']['Sales']

t_stat, p_value = stats.ttest_ind(tech_sales, furniture_sales)

print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_value:.6f}")
print(f"Result: {'Significant difference' if p_value < 0.05 else 'No significant difference'}")

### Chi-Square Test: Category vs Profit Status

In [ ]:
contingency = pd.crosstab(df['Category'], df['Profit'].apply(lambda x: 'Profitable' if x > 0 else 'Loss'))
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)

print(f"Chi-Square: {chi2:.4f}")
print(f"P-Value: {p_chi:.6f}")
print(f"Result: {'Dependent' if p_chi < 0.05 else 'Independent'}")

### 95% Confidence Interval for Sales

In [ ]:
sales_mean = df['Sales'].mean()
sales_std = df['Sales'].std()
n = len(df)
margin = stats.t.ppf(0.975, n-1) * (sales_std / np.sqrt(n))

print(f"Mean Sales: ${sales_mean:.2f}")
print(f"95% CI: [${sales_mean - margin:.2f}, ${sales_mean + margin:.2f}]")

## Day 23-24: Time Series Analysis

In [ ]:
monthly_ts = df.set_index('Order_Date').resample('M')['Sales'].sum()
monthly_ts['MA_3'] = monthly_ts.rolling(3).mean()
monthly_ts['MA_6'] = monthly_ts.rolling(6).mean()

plt.figure(figsize=(12, 6))
plt.plot(monthly_ts.index, monthly_ts['Sales'], label='Actual')
plt.plot(monthly_ts.index, monthly_ts['MA_3'], label='3-Month MA')
plt.plot(monthly_ts.index, monthly_ts['MA_6'], label='6-Month MA')
plt.legend()
plt.title('Sales Trend with Moving Averages')
plt.show()

## Customer Segmentation (K-Means Clustering)

In [ ]:
# Prepare features
customer_features = df.groupby('Customer_Name').agg({
    'Sales': ['sum', 'count'],
    'Profit': 'sum',
    'Discount': 'mean'
}).reset_index()
customer_features.columns = ['Customer', 'Total_Sales', 'Orders', 'Total_Profit', 'Avg_Discount']

X = customer_features[['Total_Sales', 'Total_Profit', 'Orders', 'Avg_Discount']].fillna(0)
X_scaled = StandardScaler().fit_transform(X)

# Find optimal K
inertias = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertias, 'bo-')
plt.xlabel('K')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

In [ ]:
# Apply K-Means with K=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
customer_features['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualize with PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))
colors = ['red', 'blue', 'green', 'purple']
for i in range(4):
    mask = customer_features['Cluster'] == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[i], label=f'Cluster {i}', alpha=0.6)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.title('Customer Segments')
plt.show()

print(customer_features.groupby('Cluster')[['Total_Sales', 'Total_Profit']].mean())

## Day 25-26: Predictive Modeling

### Linear Regression: Predict Sales

In [ ]:
X_reg = df[['Quantity', 'Discount', 'Profit']]
y_reg = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: ${mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print("\nFeature Importance:")
for feat, coef in zip(X_reg.columns, lr.coef_):
    print(f"  {feat}: {coef:.4f}")

### Logistic Regression: Predict Profitability

In [ ]:
df['Is_Profitable'] = (df['Profit'] > 0).astype(int)
X_clf = df[['Sales', 'Quantity', 'Discount']]
y_clf = df['Is_Profitable']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_c, y_train_c)
y_pred_c = log_reg.predict(X_test_c)

print(f"Accuracy: {accuracy_score(y_test_c, y_pred_c):.4f}")
print(f"Precision: {precision_score(y_test_c, y_pred_c):.4f}")
print(f"Recall: {recall_score(y_test_c, y_pred_c):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_c, y_pred_c))

## Summary

Task 4 Complete! We covered:
- ✅ Descriptive Statistics
- ✅ Hypothesis Testing (T-Test, Chi-Square)
- ✅ Confidence Intervals
- ✅ Time Series Analysis
- ✅ K-Means Clustering (4 segments)
- ✅ Linear Regression
- ✅ Logistic Regression

**Key Finding:** Quantity is the most important feature for predicting sales!